<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/01_sba_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Notebook 01: SBA Loan Data Cleaning
# Purpose: Load raw SBA 7(a) loan data,
#          filter to RGV zip codes and FY2018-2022,
#          clean for modeling, save to data/cleaned/
# ============================================

# Mount Google Drive so we can access our raw data files
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd

# Define folder paths so we don't repeat long strings everywhere
# RAW = where original untouched downloads live
# CLEANED = where filtered/cleaned data gets saved
# MERGED = where the final combined dataset goes (used in notebook 05)
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

# Make sure cleaned folder exists
os.makedirs(CLEANED, exist_ok=True)

# Verify our raw files are accessible
print("Raw files:", sorted(os.listdir(RAW)))

Mounted at /content/drive
Raw files: ['7a_504_foia_data_dictionary.xlsx', 'bds2023_sec.csv', 'bds2023_st_cty.csv', 'foia-7a-fy2010-fy2019-as-of-251231.csv', 'foia-7a-fy2020-present-as-of-251231.csv', 'zbp18detail.txt', 'zbp18detail.zip', 'zbp19detail.txt', 'zbp19detail.zip', 'zbp20detail.txt', 'zbp20detail.zip', 'zbp21detail.txt', 'zbp21detail.zip', 'zbp22detail.txt', 'zbp22detail.zip']


In [2]:
# load both sba loan files to clean and merge
sba_2010 = pd.read_csv(f'{RAW}/foia-7a-fy2010-fy2019-as-of-251231.csv', low_memory=False)
sba_2020 = pd.read_csv(f'{RAW}/foia-7a-fy2020-present-as-of-251231.csv', low_memory=False)

# Stack both files into one DataFrame since they have the same columns
# ignore_index=True resets the row numbers so they don't duplicate
sba_all = pd.concat([sba_2010, sba_2020], ignore_index=True)

# how mnay total lonas loaded and columns
print(f'Total loans loaded: {len(sba_all):,}')
print(f'Columns: {list(sba_all.columns)}')


Total loans loaded: 903,617
Columns: ['asofdate', 'program', 'l2locid', 'borrname', 'borrstreet', 'borrcity', 'borrstate', 'borrzip', 'bankname', 'bankfdicnumber', 'bankncuanumber', 'bankstreet', 'bankcity', 'bankstate', 'bankzip', 'grossapproval', 'sbaguaranteedapproval', 'approvaldate', 'approvalfiscalyear', 'firstdisbursementdate', 'processingmethod', 'subprogram', 'initialinterestrate', 'fixedorvariableinterestind', 'terminmonths', 'naicscode', 'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty', 'projectstate', 'sbadistrictoffice', 'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus', 'paidinfulldate', 'chargeoffdate', 'grosschargeoffamount', 'revolverstatus', 'jobssupported', 'collateralind', 'soldsecmrktind']


In [3]:
#Filter by state (TX) drop rest of rows
sba_all = sba_all[sba_all['borrstate'] == 'TX']

#check outcome
print(f'Total Texas loans: {len(sba_all)}')

Total Texas loans: 68967


In [4]:
# drop columns until 2019-2022
sba_all = sba_all[sba_all['approvalfiscalyear'].between(2018, 2022)]
print(f'Total Texas loans from 2018-2022: {len(sba_all)}')

Total Texas loans from 2018-2022: 18844


In [5]:
# Check year distribution to make sure filter worked right
print(sba_all['approvalfiscalyear'].value_counts().sort_index())
# we can notice here that 2020 during covide not alot of
# businesses took out SBA loans compared to rest of years

approvalfiscalyear
2018    4455
2019    3737
2020    3031
2021    4116
2022    3505
Name: count, dtype: int64


In [6]:
# check the loan status rows to see what we could use
# for prediction model
print(sba_all['loanstatus'].value_counts())

loanstatus
EXEMPT    8540
PIF       7140
CANCLD    2106
CHGOFF     979
COMMIT      79
Name: count, dtype: int64


In [7]:
# only get PIF(paid in full) and CHGOFF(business defaulted/failed)
sba_all = sba_all[sba_all['loanstatus'].isin(['PIF', 'CHGOFF'])]
print(sba_all['loanstatus'].value_counts())

loanstatus
PIF       7140
CHGOFF     979
Name: count, dtype: int64


In [8]:
sba_all['defaulted'] = (sba_all['loanstatus'] == 'CHGOFF')# True/False
sba_all['defaulted'] = sba_all['defaulted'].astype(int) # convert to 1/0

print(sba_all['defaulted'].value_counts())
print(f'Default rate: {sba_all["defaulted"].mean()*100:.1f}%')

defaulted
0    7140
1     979
Name: count, dtype: int64
Default rate: 12.1%


In [9]:
# quick visual of loans that have been defaulted in the RGV
print(sba_all[sba_all['loanstatus'] == 'CHGOFF'].head())

        asofdate program   l2locid                           borrname  \
287   12/31/2025      7A   12096.0                   BCI CONSTRUCTION   
3715  12/31/2025      7A   48270.0                 SOLE AESTHETIC LLC   
3727  12/31/2025      7A   33850.0  Alamo Relocation and Storage Inc.   
3783  12/31/2025      7A  444341.0                JOKER SOLUTIONS LLC   
3809  12/31/2025      7A   44449.0                    Link Consulting   

                borrstreet     borrcity borrstate  borrzip  \
287        511 ATKERSON LN       EULESS        TX    76040   
3715  5959 WEST LOOP South     BELLAIRE        TX    77401   
3727     1113 E Houston St  San Antonio        TX    78205   
3783  312 COUNTY ROAD 6720      NATALIA        TX    78059   
3809     1674 GLENCAIRN LN   LEWISVILLE        TX    75067   

                                       bankname  bankfdicnumber  ...  \
287       Wells Fargo Bank National Association          3511.0  ...   
3715  JPMorgan Chase Bank, National Associat

In [10]:
# Keep only the 16 columns relevant to the prediction model
# Drops administrative columns like bank info, dates, and IDs we don't need
sba_all = sba_all[['borrname', 'borrcity', 'borrzip', 'grossapproval', 'sbaguaranteedapproval',
                    'approvalfiscalyear', 'initialinterestrate', 'terminmonths', 'naicscode',
                    'naicsdescription', 'businesstype', 'businessage', 'loanstatus', 'defaulted',
                    'jobssupported', 'projectcounty']]

# Verify the result — should show 16 columns, 140 rows, and null counts per column
print(f'Columns: {len(sba_all.columns)}')
print(f'Rows: {len(sba_all)}')
print(sba_all.isnull().sum())

Columns: 16
Rows: 8119
borrname                   0
borrcity                   0
borrzip                    0
grossapproval              0
sbaguaranteedapproval      0
approvalfiscalyear         0
initialinterestrate        0
terminmonths               0
naicscode                  0
naicsdescription          90
businesstype               0
businessage              105
loanstatus                 0
defaulted                  0
jobssupported              0
projectcounty              0
dtype: int64


In [11]:
# Fill missing industry descriptions with 'Unknown' to avoid dropping rows
sba_all['naicsdescription'] = sba_all['naicsdescription'].fillna('Unknown')

# Fill missing business age with 'Unknown' — only 3 rows affected
sba_all['businessage'] = sba_all['businessage'].fillna('Unknown')

# Verify no nulls remain
print(sba_all.isnull().sum())

borrname                 0
borrcity                 0
borrzip                  0
grossapproval            0
sbaguaranteedapproval    0
approvalfiscalyear       0
initialinterestrate      0
terminmonths             0
naicscode                0
naicsdescription         0
businesstype             0
businessage              0
loanstatus               0
defaulted                0
jobssupported            0
projectcounty            0
dtype: int64


In [12]:
# Save the cleaned SBA dataset to data/cleaned/ folder
# index=False prevents pandas from writing row numbers as a column
sba_all.to_csv(f'{CLEANED}/sba_rgv_clean.csv', index=False)

# Confirm save was successful by printing shape and first few rows
print(f'Saved: {len(sba_all)} rows, {len(sba_all.columns)} columns')
print(sba_all.head())

Saved: 8119 rows, 16 columns
                     borrname  borrcity  borrzip  grossapproval  \
104  Elite Sprinkler Services      KATY    77450          15000   
114                Tax2Go LLC    Desoto    75115         125000   
123   Upstairs Circus Atx LLC    Austin    78701         455000   
200    BARGE INVESTMENTS INC.  LONGVIEW    75605         200000   
258                  PP9A LLC   BAYTOWN    77523          40000   

     sbaguaranteedapproval  approvalfiscalyear  initialinterestrate  \
104                 7500.0                2018                10.00   
114               106250.0                2018                 7.25   
123               341250.0                2018                 6.50   
200               150000.0                2018                 7.25   
258                20000.0                2018                 7.85   

     terminmonths  naicscode  \
104           120   238220.0   
114           120   541213.0   
123           120   722410.0   
200          